# 039 — Clasificación logística y umbrales

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Regresión logística:** score lineal `z = β₀ + βᵀx` pasado por la sigmoide
`p̂ = σ(z) = 1/(1+e^(−z))`. El modelo es lineal en el log-odds: `log(p/(1−p)) = z`;
cada unidad de xⱼ multiplica los odds por `e^{βⱼ}`.

**Pérdida (entropía cruzada):** `L = −(1/n)Σ[y log p̂ + (1−y) log(1−p̂)]` — convexa,
castiga sin cota la confianza equivocada, gradiente `(p̂−y)·x`. Sin solución cerrada;
con datos separables diverge salvo regularización.

**Umbral por costos:** el modelo da p̂; la decisión usa
`t* = C_FP/(C_FP + C_FN)`. Mover t reparte errores (precision↔recall), no mejora el score.

**Calibración:** p̂ ≈ 0.7 debe corresponder a ~70 % de positivos reales; se diagnostica con
el diagrama de confiabilidad y se corrige con Platt o isotónica en validación.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Sigmoide a mano.** Con el modelo `z = −2 + 0.8x`: (a) calcula p̂ para
x ∈ {0, 2.5, 5}; (b) ¿en qué x vale p̂ = 0.5? (c) ¿cuál es el odds ratio por unidad de x?

**Ejercicio 2 — Log-loss de dos modelos.** Sobre 4 casos con etiquetas y = [1, 0, 1, 0],
el modelo A predice p̂ = [0.9, 0.1, 0.8, 0.3] y el modelo B predice p̂ = [0.6, 0.4, 0.6, 0.4].
Ambos aciertan la clase en los 4 casos con t = 0.5 (misma accuracy). Calcula el log-loss
de cada uno y explica qué distingue la métrica que la accuracy no ve.

**Ejercicio 3 — Umbral por costos.** En detección de fraude, revisar manualmente una
transacción legítima (FP) cuesta 10; dejar pasar un fraude (FN) cuesta 490. (a) Calcula el
umbral óptimo t*. (b) Con p̂ = 0.05, ¿qué decides con t = 0.5 y con t*? (c) ¿Qué supone
esta fórmula sobre p̂ para ser válida?

**Ejercicio 4 — Umbral de score vs. umbral de probabilidad.** Ejecuta
`run_lab("ml", seed=39)` (celda TODO). El laboratorio elige un umbral sobre la feature
cruda maximizando accuracy con costos implícitamente simétricos. Explica: (a) qué pieza
añade la regresión logística entre la feature y la decisión, y (b) cómo recalcularías el
umbral del laboratorio si un falso negativo costara 9 veces más que un falso positivo
(pista: necesitas p̂, no la feature).


In [ ]:
# TODO: ejecuta run_lab("ml", seed=39)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicios 1 y 2
import math

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

# Ejercicio 1: p̂ en x = 0, 2.5, 5 con z = −2 + 0.8x
# Ejercicio 2: log-loss de A y B
y   = [1, 0, 1, 0]
p_a = [0.9, 0.1, 0.8, 0.3]
p_b = [0.6, 0.4, 0.6, 0.4]

def log_loss(y_true, p_hat):
    return None  # completa: −(1/n) Σ y·log(p) + (1−y)·log(1−p)


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 3: umbral por costos
c_fp, c_fn = 10, 490
t_estrella = None  # completa: C_FP / (C_FP + C_FN)
# decisión para p̂ = 0.05 con t=0.5 y con t*


## Reflexión

1. El laboratorio barre umbrales maximizando accuracy. Si el costo de un falso negativo
   fuera 9 veces el de un falso positivo, ¿qué umbral de probabilidad sería óptimo y por
   qué la accuracy dejaría de ser la métrica correcta para elegirlo?
2. Un modelo produce p̂ = 0.99 para un caso que resulta negativo. Calcula su contribución
   al log-loss y compárala con la de un p̂ = 0.6 equivocado. ¿Qué propiedad de la entropía
   cruzada ilustra la diferencia?
3. ¿Por qué "mover el umbral" nunca puede arreglar un modelo mal calibrado, y qué
   procedimiento sí lo haría sin reentrenar los coeficientes?
